<a href="https://colab.research.google.com/github/HimynameisSharon/smartenergy/blob/main/smart_energy_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Smart Energy Dashboard
> Generator & Grid Usage Analysis | Anomaly Detection | Forecasting

---
**Sections:**
1. 📦 Setup & Data Loading
2. 🔍 Exploratory Analysis
3. 🚨 Anomaly Detection
4. 🔮 Diesel Consumption Forecasting
5. 📊 Interactive Dashboard
6. 🧮 Hourly Diesel Estimator Widget
7. 📄 Auto-Generated Report


## 📦 1. Setup & Data Loading

In [ ]:
# Install required libraries
!pip install plotly ipywidgets kaleido --quiet

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# ── Plotly renderer for Colab ──────────────────────────────────────────────
import plotly.io as pio
pio.renderers.default = 'colab'

print('✅ All libraries loaded successfully')

✅ All libraries loaded successfully


In [ ]:
from io import StringIO

csv_data = """DATE,PGN 107,PGN 107,PGN 120,PGN 120,Grid start,Grid end ,Diesel start,Diesel End,Diesel Received,Shortage,Total Generator,Total Electricity,Total Diesel Consumption
M/D/Y,PGN Start,PGN end,PGN Start,PGN end,GS,GE,DS,DE,DR,S,TG,GS-GE,TDC
3/1/2025	9891	9898	0	0	29822	29840	7497	7130	0	0	7	18	367
3/2/2025	9898	9898	0	0	29840	29863	7130	7130	0	0	0	23	0
3/3/2025	9898	9918	0	0	29863	29865	7130	6197.2	0	0	20	2	932.8
3/4/2025	9918	9928	0	0	29865	29879	6197.2	5746.2	0	0	10	14	451
3/5/2025	9928	9933	0	0	29879	29898	5746.2	5505.4	0	0	5	19	240.8
3/6/2025	9933	9937	0	0	29898	29918	5505.4	5336.6	0	0	4	20	168.8
3/7/2025	9937	9945	0	0	29918	29933	5336.6	4937.2	0	0	8	15	399.4
3/8/2025	9945	9948	0	0	29933	29956	4937.2	4805.2	0	0	3	23	132
3/9/2025	9948	9951	0	0	29956	29979	4805.2	4674	0	0	3	23	131.2
3/10/2025	9951	9957	0	0	29979	29997	4674	4386	0	0	6	18	288
3/11/2025	9957	9965	0	0	29997	30013	4386	4007.2	0	0	8	16	378.8
3/12/2025	9965	9968	0	0	30013	30035	4007.2	3883.6	0	0	3	22	123.6
3/13/2025	9968	9976	1394	1394	30035	30049	3883.6	3490.6	0	0	8	14	393
3/14/2025	9976	9978	1394	1396	30049	30069	3490.6	3282.2	0	0	4	20	208.4
3/15/2025	9978	9978	1396	1396	30069	30093	3282.2	3282.2	0	0	0	24	0
3/16/2025	9978	9990	1396	1396	30093	30105	3282.2	2712	0	0	12	12	570.2
3/17/2025	9990	9991	1396	1399	30105	30125	2712	2522.2	0	0	4	20	189.8
3/18/2025	9991	9998	1399	1401	30125	30140	2522.2	4872	3000	177.2	9	15	827.4
3/19/2025	9998	10009	1401	1410	30140	30144	4872	3853.2	0	0	20	4	1018.8
3/20/2025	10009	10016	1410	1414	30144	30157	3853.2	3313	0	0	11	13	540.2
3/21/2025	10016	10017	1414	1419	30157	30177	3313	3023	0	0	6	20	290
3/22/2025	10017	10025	1419	1419	30177	30191	3023	2658	0	0	8	14	365
3/23/2025	10025	10025	1419	1424	30191	30210	2658	2439.8	0	0	5	19	218.2
3/24/2025	10025	10025	1424	1429	30210	30229	2439.8	2176.8	0	0	5	19	263
3/25/2025	10025	10025	1429	1433	30229	30249	2176.8	1973.8	0	0	4	20	203
3/26/2025	10025	10025	1433	1434	30249	30272	1973.8	1922	0	0	1	23	51.8
3/27/2025	10025	10025	1434	1444	30272	30288	1922	2151.6	800	318.8	10	16	889.2
3/28/2025	10025	10034	1444	1454	30288	30292	2151.6	12772.4	12000	379.2	19	4	1758.4
3/29/2025	10034	10043	1454	1456	30292	30305	12772.4	12187.6	0	0	11	13	584.8
3/30/2025	10043	10043	1456	1471	30305	30314	12187.6	11445.8	0	0	15	9	741.8
3/31/2025	10043	10049	1471	1472	30314	30331	11445.8	11159.4	0	0	7	17	286.4
4/1/2025	10049	10051	1472	1473	30331	30354	11159.4	11037.6	0	0	3	23	121.8
4/2/2025	10051	10051	1473	1475	30354	30377	11037.6	11037.6	0	0	2	23	0
4/3/2025	10051	10051	1475	1481	30377	30394	11037.6	10712.6	0	0	6	17	325
4/4/2025	10051	10051	1481	1486	30394	30414	10712.6	10472	0	0	5	20	240.6
4/5/2025	10051	10051	1486	1501	30414	30425	10472	9833.4	0	0	15	11	638.6
4/6/2025	10051	10051	1501	1507	30425	30444	9833.4	9557	0	0	6	19	276.4
4/7/2025	10051	10051	1507	1514	30444	30459	9557	9206	0	0	7	15	351
4/8/2025	10051	10051	1514	1522	30459	30475	9206	8819	0	0	8	16	387
4/9/2025	10051	10051	1522	1528	30475	30496	8819	8512	0	0	6	21	307
4/10/2025	10051	10051	1528	1536	30496	30511	8512	8094.6	0	0	8	15	417.4
4/11/2025	10051	10051	1536	1542	30511	30530	8094.6	7796	0	0	6	19	298.6
4/12/2025	10051	10051	1542	1550	30530	30545	7796	7386.2	0	0	8	15	409.8
4/13/2025	10051	10051	1550	1557	30545	30563	7386.2	7092.8	0	0	7	18	293.4
4/14/2025	10051	10051	1557	1563	30563	30581	7092.8	6768	0	0	6	18	324.8
4/15/2025	10051	10051	1563	1571	30581	30598	6768	6516.2	0	0	8	17	251.8
4/16/2025	10051	10051	1571	1574	30598	30618	6516.2	6374	0	0	3	20	142.2
4/17/2025	10051	10051	1574	1574	30618	30642	6374	6374	0	0	0	24	0
4/18/2025	10051	10053	1574	1578	30642	30661	6374	6058	0	0	6	19	316
4/19/2025	10053	10055	1578	1585	30661	30676	6058	5609	0	0	9	15	449
4/20/2025	10055	10055	1585	1593	30676	30694	5609	5236.4	0	0	8	18	372.6
4/21/2025	10055	10055	1593	1603	30694	30712	5236.4	4772.8	0	0	10	18	463.6
4/22/2025	10055	10055	1603	1607	30712	30732	4772.8	4578	0	0	4	20	194.8
4/23/2025	10055	10055	1607	1609	30732	30742	4578	4481.4	0	0	2	10	96.6
4/24/2025	10055	10055	1609	1615	30742	30773	4481.4	4195	0	0	6	31	286.4
4/25/2025	10055	10055	1615	1624	30773	30788	4195	3762	0	0	9	15	433
4/26/2025	10055	10055	1624	1631	30788	30806	3762	3429.8	0	0	7	18	332.2
4/27/2025	10055	10055	1631	1642	30806	30818	3429.8	2938	0	0	11	12	491.8
4/28/2025	10055	10055	1642	1651	30818	30833	2938	2469	0	0	9	15	469
5/1/2025	10055	10055	1661	1664	30872	30894	1973.8	1850.2	0	0	3	22	123.6
5/2/2025	10055	10055	1664	1665	30894	30917	1850.2	1800.2	0	0	1	23	50
5/3/2025	10055	10055	1665	1669	30917	30940	1800.2	1609.2	0	0	4	23	191
5/4/2025	10055	10055	1669	1669	30940	30963	1609.2	1609.2	0	0	0	23	0
5/5/2025	10055	10055	1669	1673	30963	30985	1609.2	1401	0	0	4	22	208.2
5/6/2025	10055	10055	1673	1673	30985	31007	1401	1401	0	0	0	22	0
5/7/2025	10055	10055	1673	1678	31007	31028	1401	1140	0	0	5	21	261
5/8/2025	10055	10055	1678	1678	31028	31052	1140	1140	0	0	0	24	0
5/9/2025	10055	10055	1678	1684	31052	31070	1140	2685	2000	146	6	18	309
5/10/2025	10055	10055	1684	1684	31070	31094	2685	2685	0	0	0	24	0
5/11/2025	10055	10055	1684	1690	31094	31111	2685	2413.6	0	0	6	17	271.4
5/12/2025	10055	10055	1690	1697	31111	31128	2413.6	2076	0	0	7	17	337.6
5/13/2025	10055	10055	1697	1705	31128	31144	2076	1655.6	0	0	8	16	420.4
5/14/2025	10055	10055	1705	1715	31144	31158	1655.6	1160.8	0	0	10	14	494.8
5/15/2025	10055	10055	1715	1719	31158	31178	1160.8	3762	3000	398.8	4	20	0
5/16/2025	10055	10055	1719	1719	31178	31202	3762	3762	0	0	0	24	0
5/17/2025	10055	10055	1719	1721	31202	31224	3762	3671	0	0	2	22	91
5/18/2025	10055	10055	1721	1728	31224	31241	3671	3313	0	0	7	17	358
5/19/2025	10055	10055	1728	1731	31241	31262	3313	3167	0	0	3	21	146
5/20/2025	10055	10055	1731	1733	31262	31284	3167	3079.8	0	0	2	22	87.2
5/21/2025	10055	10055	1733	1756	31284	31286	3079.8	1949	0	0	23	2	1130.8
5/22/2025	10055	10055	1756	1780	31286	31286	1949	838	0	0	24	0	1111
5/23/2025	10055	10067	1780	1790	31286	31286	838	4576	5000	246.8	22	0	1015.2
5/24/2025	10067	10090	1790	1790	31286	31286	4576	22601	20000	988.4	23	0	986.6
5/25/2025	10090	10107	1790	1798	31286	31286	22601	21641.4	0	0	25	0	959.6
5/26/2025	10107	10119	1798	1810	31286	31286	21641.4	20503.8	0	0	24	0	1137.6
5/27/2025	10119	10130	1810	1822	31286	31286	20503.8	19419	0	0	23	0	1084.8
5/28/2025	10130	10130	1822	1837	31286	31298	19419	18654.4	0	0	15	12	764.6
5/29/2025	10130	10130	1837	1858	31298	31300	18654.4	17715.8	0	0	21	2	938.6
5/30/2025	10130	10132	1858	1869	31300	31313	17715.8	17137.8	0	0	13	13	578
5/31/2025	10132	10133	1869	1882	31313	31322	17137.8	16520	0	0	14	9	617.8"""

column_names = [
    'DATE', 'PGN_107_Start', 'PGN_107_End',
    'PGN_120_Start', 'PGN_120_End',
    'Grid_Start', 'Grid_End',
    'Diesel_Start', 'Diesel_End',
    'Diesel_Received', 'Shortage',
    'Total_Generator', 'Total_Electricity', 'Total_Diesel_Consumption'
]

df = pd.read_csv(StringIO(csv_data), sep='\t', names=column_names, skiprows=2, parse_dates=['DATE'])
df.set_index('DATE', inplace=True)
df.sort_index(inplace=True)

# ── Derived columns ────────────────────────────────────────────────────────
df['PGN_107_kWh'] = df['PGN_107_End'] - df['PGN_107_Start']
df['PGN_120_kWh'] = df['PGN_120_End'] - df['PGN_120_Start']
df['Grid_Hours']  = df['Grid_End'] - df['Grid_Start']
df['day_of_week'] = df.index.dayofweek
df['month_name']  = df.index.strftime('%b %Y')
df['Diesel_Stock'] = df['Diesel_End']  # tank level at end of day
df['Has_Shortage'] = df['Shortage'] > 0

df.to_csv('company_generator_data.csv')
print(f'✅ Data loaded: {len(df)} days  |  {df.index.min().date()} → {df.index.max().date()}')
df.head(3)

✅ Data loaded: 90 days  |  2025-03-01 → 2025-05-31


,PGN_107_Start,PGN_107_End,PGN_120_Start,PGN_120_End,Grid_Start,Grid_End,Diesel_Start,Diesel_End,Diesel_Received,Shortage,Total_Generator,Total_Electricity,Total_Diesel_Consumption,PGN_107_kWh,PGN_120_kWh,Grid_Hours,day_of_week,month_name,Diesel_Stock,Has_Shortage
DATE,,,,,,,,,,,,,,,,,,,,
2025-03-01,9891,9898,0,0,29822,29840,7497.0,7130.0,0,0.0,7,18,367.0,7,0,18,5,Mar 2025,7130.0,False
2025-03-02,9898,9898,0,0,29840,29863,7130.0,7130.0,0,0.0,0,23,0.0,0,0,23,6,Mar 2025,7130.0,False
2025-03-03,9898,9918,0,0,29863,29865,7130.0,6197.2,0,0.0,20,2,932.8,20,0,2,0,Mar 2025,6197.2,False


## 🔍 2. Exploratory Analysis

In [ ]:
# ── KPI summary ────────────────────────────────────────────────────────────
total_diesel   = df['Total_Diesel_Consumption'].sum()
avg_diesel     = df[df['Total_Diesel_Consumption'] > 0]['Total_Diesel_Consumption'].mean()
total_elec     = df['Total_Electricity'].sum()
total_gen_hrs  = df['Total_Generator'].sum()
total_grid_hrs = df['Total_Electricity'].sum()
shortage_days  = df['Has_Shortage'].sum()
total_received = df['Diesel_Received'].sum()
current_stock  = df['Diesel_Stock'].iloc[-1]

print('='*55)
print('          ⚡ ENERGY USAGE — KEY METRICS')
print('='*55)
print(f'  Period          : {df.index.min().date()} to {df.index.max().date()}')
print(f'  Total Days      : {len(df)}')
print(f'  Total Diesel    : {total_diesel:,.1f} L')
print(f'  Avg Daily Diesel: {avg_diesel:.1f} L  (non-zero days)')
print(f'  Total Grid Power: {total_elec:.0f} kWh')
print(f'  Total Gen Hours : {total_gen_hrs} hrs')
print(f'  Diesel Received : {total_received:,.0f} L')
print(f'  Current Stock   : {current_stock:,.1f} L')
print(f'  Shortage Days   : {shortage_days}')
print('='*55)

          ⚡ ENERGY USAGE — KEY METRICS
  Period          : 2025-03-01 to 2025-05-31
  Total Days      : 90
  Total Diesel    : 35,377.0 L
  Avg Daily Diesel: 442.2 L  (non-zero days)
  Total Grid Power: 1461 kWh
  Total Gen Hours : 720 hrs
  Diesel Received : 45,800 L
  Current Stock   : 16,520.0 L
  Shortage Days   : 7


In [ ]:
# ── Diesel consumption + grid hours dual-axis time series ──────────────────
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=('Diesel Consumption Over Time', 'Generator vs Grid Hours'),
    vertical_spacing=0.08
)

fig.add_trace(go.Scatter(
    x=df.index, y=df['Total_Diesel_Consumption'],
    name='Diesel (L)', fill='tozeroy',
    line=dict(color='#F59E0B', width=2),
    fillcolor='rgba(245,158,11,0.15)'
), row=1, col=1)

# Mark diesel refill days
refill_days = df[df['Diesel_Received'] > 0]
fig.add_trace(go.Scatter(
    x=refill_days.index, y=refill_days['Total_Diesel_Consumption'],
    mode='markers', name='Refill Day',
    marker=dict(color='#10B981', size=10, symbol='triangle-up')
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df.index, y=df['Total_Generator'],
    name='Generator (hrs)', marker_color='#EF4444', opacity=0.7
), row=2, col=1)

fig.add_trace(go.Bar(
    x=df.index, y=df['Total_Electricity'],
    name='Grid (hrs)', marker_color='#3B82F6', opacity=0.7
), row=2, col=1)

fig.update_layout(
    height=600, barmode='stack',
    template='plotly_dark',
    title='📊 Energy Overview',
    legend=dict(orientation='h', y=-0.05)
)
fig.show()

In [ ]:
# ── PGN 107 vs PGN 120 comparison ─────────────────────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('PGN 107 Daily kWh', 'PGN 120 Daily kWh')
)

fig.add_trace(go.Scatter(
    x=df.index, y=df['PGN_107_kWh'],
    fill='tozeroy', name='PGN 107',
    line=dict(color='#8B5CF6')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df['PGN_120_kWh'],
    fill='tozeroy', name='PGN 120',
    line=dict(color='#06B6D4')
), row=1, col=2)

fig.update_layout(
    height=350, template='plotly_dark',
    title='🔌 Generator Meter Comparison (PGN 107 vs PGN 120)',
)
fig.show()

print(f'PGN 107 total kWh : {df["PGN_107_kWh"].sum():,.0f}')
print(f'PGN 120 total kWh : {df["PGN_120_kWh"].sum():,.0f}')
print(f'PGN 120 active from: {df[df["PGN_120_kWh"] > 0].index.min().date()}')

PGN 107 total kWh : 242
PGN 120 total kWh : 478
PGN 120 active from: 2025-03-14


In [ ]:
# ── Diesel tank level (stock) over time ───────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df['Diesel_Stock'],
    fill='tozeroy', name='Tank Level (L)',
    line=dict(color='#10B981', width=2),
    fillcolor='rgba(16,185,129,0.15)'
))

# Mark shortage days
shortage_df = df[df['Has_Shortage']]
fig.add_trace(go.Scatter(
    x=shortage_df.index, y=shortage_df['Diesel_Stock'],
    mode='markers', name='Shortage Day',
    marker=dict(color='#EF4444', size=12, symbol='x')
))

# Mark refill events
fig.add_trace(go.Scatter(
    x=refill_days.index, y=refill_days['Diesel_Stock'],
    mode='markers', name=f'Refill ({len(refill_days)} events)',
    marker=dict(color='#F59E0B', size=10, symbol='triangle-up')
))

fig.update_layout(
    height=400, template='plotly_dark',
    title='🛢️ Diesel Tank Level Over Time',
    yaxis_title='Litres',
    legend=dict(orientation='h')
)
fig.show()

In [ ]:
# ── Day-of-week patterns ───────────────────────────────────────────────────
day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_diesel = df.groupby('day_of_week')['Total_Diesel_Consumption'].mean()
dow_elec   = df.groupby('day_of_week')['Total_Electricity'].mean()
dow_gen    = df.groupby('day_of_week')['Total_Generator'].mean()

fig = make_subplots(rows=1, cols=3,
    subplot_titles=('Avg Diesel by Day','Avg Grid Hours by Day','Avg Gen Hours by Day')
)

colors = ['#EF4444' if i < 5 else '#6B7280' for i in range(7)]

fig.add_trace(go.Bar(x=[day_names[i] for i in dow_diesel.index],
    y=dow_diesel.values, marker_color=colors, name='Diesel'), row=1, col=1)
fig.add_trace(go.Bar(x=[day_names[i] for i in dow_elec.index],
    y=dow_elec.values, marker_color='#3B82F6', name='Grid'), row=1, col=2)
fig.add_trace(go.Bar(x=[day_names[i] for i in dow_gen.index],
    y=dow_gen.values, marker_color='#F59E0B', name='Generator'), row=1, col=3)

fig.update_layout(
    height=380, template='plotly_dark',
    title='📅 Day-of-Week Usage Patterns  (grey = weekend)',
    showlegend=False
)
fig.show()

## 🚨 3. Anomaly Detection

In [ ]:
# ── IQR-based anomaly detection ────────────────────────────────────────────
# Impute weekday zeros first
df_clean = df.copy()
median_nz = df_clean[df_clean['Total_Diesel_Consumption'] > 0]['Total_Diesel_Consumption'].median()
weekday_zero = (df_clean['day_of_week'].isin(range(5))) & (df_clean['Total_Diesel_Consumption'] == 0)
df_clean.loc[weekday_zero, 'Total_Diesel_Consumption'] = median_nz
print(f'Imputed {weekday_zero.sum()} weekday zeros with median ({median_nz:.1f} L)')

Q1 = df_clean['Total_Diesel_Consumption'].quantile(0.25)
Q3 = df_clean['Total_Diesel_Consumption'].quantile(0.75)
IQR = Q3 - Q1
lower_b = Q1 - 1.5 * IQR
upper_b = Q3 + 1.5 * IQR

df_clean['is_anomaly'] = (
    (df_clean['Total_Diesel_Consumption'] < lower_b) |
    (df_clean['Total_Diesel_Consumption'] > upper_b)
)
anomalies = df_clean[df_clean['is_anomaly']]

print(f'\nIQR bounds  : {lower_b:.1f} L  –  {upper_b:.1f} L')
print(f'Anomalies   : {len(anomalies)}')
print(f'HIGH spikes : {(anomalies["Total_Diesel_Consumption"] > upper_b).sum()}')
print(f'LOW dips    : {(anomalies["Total_Diesel_Consumption"] < lower_b).sum()}')

Imputed 6 weekday zeros with median (344.3 L)

IQR bounds  : -202.0 L  –  899.0 L
Anomalies   : 11
HIGH spikes : 11
LOW dips    : 0


In [ ]:
# ── Anomaly visualisation ──────────────────────────────────────────────────
fig = go.Figure()

fig.add_hrect(y0=lower_b, y1=upper_b,
    fillcolor='rgba(16,185,129,0.07)', line_width=0,
    annotation_text='Normal range', annotation_position='top left')

fig.add_trace(go.Scatter(
    x=df_clean.index, y=df_clean['Total_Diesel_Consumption'],
    mode='lines+markers', name='Diesel (L)',
    line=dict(color='#60A5FA', width=1.5),
    marker=dict(size=4)
))

fig.add_trace(go.Scatter(
    x=anomalies.index, y=anomalies['Total_Diesel_Consumption'],
    mode='markers', name=f'Anomaly ({len(anomalies)})',
    marker=dict(color='#EF4444', size=12, symbol='circle-open', line=dict(width=2))
))

for _, row in anomalies.iterrows():
    fig.add_annotation(
        x=row.name, y=row['Total_Diesel_Consumption'],
        text=f"{row['Total_Diesel_Consumption']:.0f}L",
        showarrow=True, arrowhead=1, yshift=12,
        font=dict(size=10, color='#EF4444')
    )

fig.update_layout(
    height=480, template='plotly_dark',
    title=f'🚨 Diesel Anomaly Detection  ({len(anomalies)} flagged)',
    yaxis_title='Litres', xaxis_title='Date'
)
fig.show()

In [ ]:
# ── Anomaly detail table ───────────────────────────────────────────────────
if not anomalies.empty:
    tbl = anomalies[['Total_Diesel_Consumption','Total_Electricity']].copy()
    tbl['Type'] = tbl['Total_Diesel_Consumption'].apply(
        lambda x: '🔴 HIGH' if x > upper_b else '🔵 LOW'
    )
    tbl.index = tbl.index.strftime('%Y-%m-%d')
    tbl.columns = ['Diesel (L)', 'Electricity (kWh)', 'Type']
    display(tbl)

,Diesel (L),Electricity (kWh),Type
DATE,,,
2025-03-03,932.8,2,🔴 HIGH
2025-03-19,1018.8,4,🔴 HIGH
2025-03-28,1758.4,4,🔴 HIGH
2025-05-21,1130.8,2,🔴 HIGH
2025-05-22,1111.0,0,🔴 HIGH
2025-05-23,1015.2,0,🔴 HIGH
2025-05-24,986.6,0,🔴 HIGH
2025-05-25,959.6,0,🔴 HIGH
2025-05-26,1137.6,0,🔴 HIGH


## 🔮 4. Diesel Consumption Forecasting

In [ ]:
def build_features(df_in, target='Total_Diesel_Consumption', lags=7):
    d = df_in.copy()
    for i in range(1, lags + 1):
        d[f'lag_{i}'] = d[target].shift(i)
    d['rolling_7']   = d[target].shift(1).rolling(7).mean()
    d['rolling_7_std'] = d[target].shift(1).rolling(7).std()
    d['dow']         = d.index.dayofweek
    d['dom']         = d.index.day
    d['month']       = d.index.month
    d['is_weekend']  = d['dow'].isin([5, 6]).astype(int)
    return d.dropna()

df_feat = build_features(df_clean)
feat_cols = [c for c in df_feat.columns
             if c not in ['Total_Diesel_Consumption','Total_Electricity',
                          'is_anomaly','Has_Shortage','month_name','day_of_week']]

X = df_feat[feat_cols]
y = df_feat['Total_Diesel_Consumption']
split = int(len(X) * 0.8)
X_tr, X_te = X.iloc[:split], X.iloc[split:]
y_tr, y_te = y.iloc[:split], y.iloc[split:]

model = RandomForestRegressor(n_estimators=200, max_depth=10,
                               min_samples_split=3, random_state=42)
model.fit(X_tr, y_tr)
preds = model.predict(X_te)

mae  = mean_absolute_error(y_te, preds)
rmse = np.sqrt(mean_squared_error(y_te, preds))
r2   = r2_score(y_te, preds)
mape = np.mean(np.abs((y_te - preds) / np.where(y_te == 0, 1, y_te))) * 100

print(f'Test set performance  ({len(y_te)} days):')
print(f'  MAE   = {mae:.1f} L')
print(f'  RMSE  = {rmse:.1f} L')
print(f'  MAPE  = {mape:.1f}%')
print(f'  R²    = {r2:.3f}')

Test set performance  (17 days):
  MAE   = 95.1 L
  RMSE  = 116.0 L
  MAPE  = 21.1%
  R²    = 0.907


In [ ]:
# ── Actual vs Predicted ────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_te.index, y=y_te.values,
    mode='lines+markers', name='Actual',
    line=dict(color='#60A5FA', width=2)))
fig.add_trace(go.Scatter(x=y_te.index, y=preds,
    mode='lines+markers', name='Predicted',
    line=dict(color='#F59E0B', width=2, dash='dot')))
fig.update_layout(
    height=400, template='plotly_dark',
    title=f'🤖 Model: Actual vs Predicted  (MAE={mae:.1f}L, R²={r2:.2f})',
    yaxis_title='Diesel (L)'
)
fig.show()

In [ ]:
# ── 7-day recursive forecast ───────────────────────────────────────────────
def recursive_forecast(model, last_row, feat_cols, days=7):
    max_lags = max([int(c.split('_')[-1]) for c in feat_cols if c.startswith('lag_')], default=0)
    feats = last_row[feat_cols].copy()
    results = []
    for d in range(days):
        pred = model.predict(pd.DataFrame([feats.values], columns=feat_cols))[0]
        results.append(pred)
        for i in range(max_lags, 1, -1):
            if f'lag_{i}' in feat_cols and f'lag_{i-1}' in feat_cols:
                feats[f'lag_{i}'] = feats[f'lag_{i-1}']
        if 'lag_1' in feat_cols:
            feats['lag_1'] = pred
        nxt = last_row.name + pd.Timedelta(days=d+1)
        for col, val in [('dow', nxt.dayofweek), ('dom', nxt.day),
                          ('month', nxt.month),
                          ('is_weekend', int(nxt.dayofweek in [5,6]))]:
            if col in feat_cols:
                feats[col] = val
    return results

future_preds = recursive_forecast(model, df_feat.iloc[-1], feat_cols, days=7)
future_dates = pd.date_range(start=df_feat.index[-1] + pd.Timedelta(days=1), periods=7)

# Confidence band (±1 MAE)
fig = go.Figure()
hist_window = df_clean['Total_Diesel_Consumption'].iloc[-30:]
fig.add_trace(go.Scatter(
    x=hist_window.index, y=hist_window.values,
    mode='lines', name='Historical (30d)',
    line=dict(color='#60A5FA', width=2)
))
fig.add_trace(go.Scatter(
    x=list(future_dates) + list(future_dates[::-1]),
    y=[p + mae for p in future_preds] + [p - mae for p in future_preds[::-1]],
    fill='toself', fillcolor='rgba(245,158,11,0.15)',
    line=dict(color='rgba(0,0,0,0)'), name='±MAE band'
))
fig.add_trace(go.Scatter(
    x=future_dates, y=future_preds,
    mode='lines+markers', name='7-Day Forecast',
    line=dict(color='#F59E0B', width=2.5, dash='dot'),
    marker=dict(size=8)
))
fig.update_layout(
    height=420, template='plotly_dark',
    title='🔮 7-Day Diesel Consumption Forecast',
    yaxis_title='Litres'
)
fig.show()

fdf = pd.DataFrame({'Date': future_dates.strftime('%Y-%m-%d'),
                    'Forecast (L)': [round(p,1) for p in future_preds]})
print('\n📅 7-Day Forecast Table:')
print(fdf.to_string(index=False))
print(f'\n  Total projected: {sum(future_preds):,.0f} L over 7 days')


📅 7-Day Forecast Table:
      Date  Forecast (L)
2025-06-01         750.8
2025-06-02         744.4
2025-06-03         747.6
2025-06-04         747.6
2025-06-05         747.9
2025-06-06         742.7
2025-06-07         742.7

  Total projected: 5,224 L over 7 days


In [ ]:
# ── Feature importance ─────────────────────────────────────────────────────
fi = pd.DataFrame({'Feature': feat_cols,
                   'Importance': model.feature_importances_})\
       .sort_values('Importance', ascending=True).tail(12)

fig = go.Figure(go.Bar(
    x=fi['Importance'], y=fi['Feature'],
    orientation='h',
    marker=dict(color=fi['Importance'], colorscale='Oranges')
))
fig.update_layout(
    height=380, template='plotly_dark',
    title='🎯 Top Feature Importances',
    xaxis_title='Importance'
)
fig.show()

## 📊 5. Interactive Dashboard

In [ ]:
# ── Full multi-panel dashboard ─────────────────────────────────────────────
anomaly_count = len(anomalies)
normal_count  = len(df_clean) - anomaly_count

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Diesel & Grid Power Over Time',
        'Diesel Stock / Tank Level',
        'Anomaly Distribution',
        'Generator vs Grid Split',
        'Diesel vs Grid Correlation',
        '7-Day Forecast'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    specs=[
        [{}, {}],
        [{}, {}],
        [{"type": "scatter"}, {}]
    ]
)

# ── [1,1] Diesel & grid over time ──
fig.add_trace(go.Scatter(
    x=df_clean.index, y=df_clean['Total_Diesel_Consumption'],
    name='Diesel (L)', line=dict(color='#F59E0B'),
    fill='tozeroy', fillcolor='rgba(245,158,11,0.1)'
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df.index, y=df['Total_Electricity'],
    name='Grid (hrs)', line=dict(color='#3B82F6'),
    yaxis='y2'
), row=1, col=1)

# ── [1,2] Diesel stock ──
fig.add_trace(go.Scatter(
    x=df.index, y=df['Diesel_Stock'],
    name='Tank Level', fill='tozeroy',
    line=dict(color='#10B981'),
    fillcolor='rgba(16,185,129,0.12)'
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=refill_days.index, y=refill_days['Diesel_Stock'],
    mode='markers', name='Refill',
    marker=dict(color='#F59E0B', size=9, symbol='triangle-up'),
    showlegend=False
), row=1, col=2)

# ── [2,1] Anomaly pie ──
fig.add_trace(go.Bar(
    x=['Normal', 'Anomaly'],
    y=[normal_count, anomaly_count],
    marker_color=['#10B981', '#EF4444'],
    showlegend=False, name='Anomaly'
), row=2, col=1)

# ── [2,2] Monthly generator vs grid stacked ──
monthly = df.groupby('month_name')[['Total_Generator','Total_Electricity']].sum()
fig.add_trace(go.Bar(
    x=monthly.index, y=monthly['Total_Generator'],
    name='Generator hrs', marker_color='#EF4444'
), row=2, col=2)
fig.add_trace(go.Bar(
    x=monthly.index, y=monthly['Total_Electricity'],
    name='Grid hrs', marker_color='#3B82F6'
), row=2, col=2)

# ── [3,1] Scatter diesel vs electricity ──
fig.add_trace(go.Scatter(
    x=df_clean['Total_Diesel_Consumption'],
    y=df_clean['Total_Electricity'],
    mode='markers',
    marker=dict(color=df_clean.index.month, colorscale='Viridis', size=7, opacity=0.7),
    name='Diesel vs Grid'
), row=3, col=1)

# ── [3,2] Forecast ──
fig.add_trace(go.Scatter(
    x=df_clean.index[-20:],
    y=df_clean['Total_Diesel_Consumption'].values[-20:],
    line=dict(color='#60A5FA'), name='Historical', showlegend=False
), row=3, col=2)
fig.add_trace(go.Scatter(
    x=future_dates, y=future_preds,
    line=dict(color='#F59E0B', dash='dot'),
    mode='lines+markers',
    name='Forecast', showlegend=False
), row=3, col=2)

fig.update_layout(
    height=950,
    template='plotly_dark',
    title=dict(
        text='⚡ Smart Energy — Full Dashboard',
        font=dict(size=22)
    ),
    barmode='stack',
    legend=dict(orientation='h', y=-0.03)
)

fig.show()

## 🧮 6. Hourly Diesel Estimator Widget

In [ ]:
# ── Interactive ipywidget (no blocking input loop) ─────────────────────────
avg_daily = df_clean[df_clean['Total_Diesel_Consumption'] > 0]['Total_Diesel_Consumption'].mean()
avg_gen_hrs_per_day = df[df['Total_Generator'] > 0]['Total_Generator'].mean()
hourly_rate = avg_daily / avg_gen_hrs_per_day

hours_slider = widgets.FloatSlider(
    value=8, min=1, max=24, step=0.5,
    description='Hours:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px')
)
output_label = widgets.HTML()

def on_change(change):
    h = change['new']
    est = h * hourly_rate
    week = est * 7
    month = est * 30
    color = '#EF4444' if est > avg_daily else '#10B981'
    output_label.value = f"""
    <div style="font-family:monospace; background:#1F2937; padding:16px;
                border-radius:8px; color:#F9FAFB; margin-top:10px;">
      <b>Avg hourly rate:</b> {hourly_rate:.2f} L/hr
      (based on {avg_gen_hrs_per_day:.1f} gen hrs/day avg)<br><br>
      <span style="font-size:1.4em; color:{color};">⚡ {est:.1f} L</span>
      <span style="color:#9CA3AF;"> for {h:.1f} hours</span><br>
      <small>7-day projection: <b>{week:.0f} L</b> &nbsp;|&nbsp;
             30-day projection: <b>{month:.0f} L</b></small>
    </div>
    """

hours_slider.observe(on_change, names='value')
on_change({'new': hours_slider.value})

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#F9FAFB">🧮 Hourly Diesel Estimator</h3>'),
    hours_slider,
    output_label
]))

## 📄 7. Auto-Generated Analysis Report

In [ ]:
corr = df_clean[['Total_Diesel_Consumption','Total_Electricity']].corr().iloc[0,1]
worst_day = df_clean['Total_Diesel_Consumption'].idxmax()
best_day  = df_clean[df_clean['Total_Diesel_Consumption'] > 0]['Total_Diesel_Consumption'].idxmin()
pgn107_share = df['PGN_107_kWh'].sum()
pgn120_share = df['PGN_120_kWh'].sum()
pgn120_pct   = pgn120_share / (pgn107_share + pgn120_share) * 100 if (pgn107_share + pgn120_share) > 0 else 0

report = f"""
╔══════════════════════════════════════════════════════════╗
║           ⚡ SMART ENERGY — ANALYSIS REPORT              ║
╚══════════════════════════════════════════════════════════╝

  Period        : {df.index.min().date()} → {df.index.max().date()}
  Days analysed : {len(df)}

── DIESEL ─────────────────────────────────────────────────
  Total consumed     : {total_diesel:,.1f} L
  Avg daily (active) : {avg_diesel:.1f} L
  Avg hourly rate    : {hourly_rate:.2f} L/hr
  Peak day           : {worst_day.date()}  ({df_clean.loc[worst_day,'Total_Diesel_Consumption']:.1f} L)
  Lowest active day  : {best_day.date()}  ({df_clean.loc[best_day,'Total_Diesel_Consumption']:.1f} L)
  Total received     : {total_received:,.0f} L  ({len(refill_days)} refill events)
  Current stock      : {current_stock:,.1f} L

── GRID & GENERATOR ───────────────────────────────────────
  Total grid power   : {total_elec:.0f} kWh
  Total gen hours    : {df['Total_Generator'].sum()} hrs
  PGN 107 output     : {pgn107_share:,.0f} kWh ({100-pgn120_pct:.1f}%)
  PGN 120 output     : {pgn120_share:,.0f} kWh ({pgn120_pct:.1f}%)
  PGN 120 online from: {df[df['PGN_120_kWh']>0].index.min().date()}

── ANOMALIES ──────────────────────────────────────────────
  Flagged days       : {anomaly_count}  ({anomaly_count/len(df)*100:.1f}% of period)
  HIGH-usage spikes  : {(anomalies['Total_Diesel_Consumption'] > upper_b).sum()}
  LOW-usage dips     : {(anomalies['Total_Diesel_Consumption'] < lower_b).sum()}
  Shortage events    : {shortage_days}

── FORECAST ───────────────────────────────────────────────
  7-day projection   : {sum(future_preds):,.0f} L
  Model R²           : {r2:.3f}   MAE: {mae:.1f} L   MAPE: {mape:.1f}%

── CORRELATION ────────────────────────────────────────────
  Diesel ↔ Grid      : {corr:.3f}  {'(inverse: more diesel = less grid, expected)' if corr < 0 else '(positive)'}

══════════════════════════════════════════════════════════
"""
print(report)


╔══════════════════════════════════════════════════════════╗
║           ⚡ SMART ENERGY — ANALYSIS REPORT              ║
╚══════════════════════════════════════════════════════════╝

  Period        : 2025-03-01 → 2025-05-31
  Days analysed : 90

── DIESEL ─────────────────────────────────────────────────
  Total consumed     : 35,377.0 L
  Avg daily (active) : 442.2 L
  Avg hourly rate    : 49.59 L/hr
  Peak day           : 2025-03-28  (1758.4 L)
  Lowest active day  : 2025-05-02  (50.0 L)
  Total received     : 45,800 L  (7 refill events)
  Current stock      : 16,520.0 L

── GRID & GENERATOR ───────────────────────────────────────
  Total grid power   : 1461 kWh
  Total gen hours    : 720 hrs
  PGN 107 output     : 242 kWh (33.6%)
  PGN 120 output     : 478 kWh (66.4%)
  PGN 120 online from: 2025-03-14

── ANOMALIES ──────────────────────────────────────────────
  Flagged days       : 11  (12.2% of period)
  HIGH-usage spikes  : 11
  LOW-usage dips     : 0
  Shortage events    : 7
